In [ ]:
from utils.spark_utils import get_spark
from pyspark.sql.functions import window, avg

spark = get_spark()


In [ ]:
df = spark.readStream.table("workspace.cryptoinsight.silver_stream")

df_gold = df.groupBy(
    "coin",
    window("ingestion_time", "1 minute")
).agg(avg("price").alias("avg_price"))

query = df_gold.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", "/Volumes/workspace/cryptoinsight/gold_stream_checkpoint/") \
    .toTable("workspace.cryptoinsight.gold_stream")

query.awaitTermination()